# 02 · Practical A — The Trading Simulator

**Read first:** Chapter 3 (*Introduction to Trading*), then Practical A.

---

## What you'll be able to do after this

- Explain, with a number attached, why a price taker is structurally negative carry and a price maker is not.
- Run a trading session tick by tick and read the position and P&L that fall out of it.
- Measure the cost of over-trading across a thousand sessions instead of arguing about it.

## The intuition, before the maths

There is one idea in this practical and it fits in a sentence:

> **If you initiate the trade, you pay. If someone else initiates, you get paid.**

Everything else follows.

### The apple market again

Apples trade at 10p. A dealer quotes you **9p bid / 11p offer** — she'll buy at 9, sell at 11.

You want an apple, so you pay 11p. Now you own an apple, and the mid is still 10p. **You are down 1p the instant you trade**, before anything has moved. If you immediately changed your mind and sold it back, you'd get 9p and be down 2p on a round trip in a market that never budged.

Now flip it. You're the dealer. Someone pays your 11p offer and someone else gives your 9p bid. You've bought at 9 and sold at 11 with no net position: **up 2p**, for having been willing to quote.

That's the whole thing. Half the spread, every trade, one direction or the other.

### Why the price maker's job is still hard

If it were just "quote a spread and collect", everyone would do it. The catch is in Chapter 3's scenario 3: the two counterparties don't always offset. Sometimes three sellers arrive in a row and you're long three apples in a market heading down. You collected 1.5p of spread and you're carrying a position you never chose.

So:

- **The price taker** controls their position and pays for the privilege.
- **The price maker** gets paid but inherits whatever position the flow hands them.

Neither is free. The practical lets you feel both.

## The maths, derived not asserted

Barely any, which is why this practical comes early.

### The two-way price

$$\text{bid} = \text{mid} - \frac{s}{2}, \qquad \text{offer} = \text{mid} + \frac{s}{2}$$

with $s$ the total bid–offer spread. The quantity that matters throughout is the **half spread**, $s/2$ — the distance from mid to either side.

### Position P&L

From Chapter 1, applied one tick at a time:

$$\Delta\text{P\&L}_{CCY2} = \text{Position}_{CCY1} \times \Delta S$$

Long and positive, or short and negative, makes money.

### The trade itself

$$\text{price taking:}\quad \text{P\&L} \mathrel{-}= \frac{s}{2} \quad\text{(buy \emph{or} sell)}$$
$$\text{price making:}\quad \text{P\&L} \mathrel{+}= \frac{s}{2} \quad\text{(bought \emph{or} sold from)}$$

**The sign does not depend on direction.** That asymmetry — between *who initiated*, not *which way* — is the entire lesson.

### Expected P&L of trading with no view

If you trade $n$ times at random in a market that is a fair coin:

$$\mathbb{E}[\text{P\&L}] = \underbrace{0}_{\text{no view, no edge}} - \underbrace{n \cdot \frac{s}{2}}_{\text{spread crossed}}$$

Linear in $n$, and always negative. There's no volume at which it turns around. Chapter 3's warning — *don't over-trade when there is spread cross involved* — is this equation.

### The tick ordering, and why it matters

Practical A's Task B fixes an order that has to be reproduced exactly:

1. Draw the spot increment.
2. **Mark the existing position to it.**
3. Move spot, advance the step.
4. Recompute bid and offer.
5. Process the trade, charging half the spread.
6. Reset the action.

Step 2 before step 5 means a trade placed *this* tick does not earn *this* tick's move. Reverse them and every trade gets a free tick of drift — your P&L quietly diverges from reality in a way that looks like skill. `tests/test_simulator.py::TestSimulatorTickOrdering` pins it down.

## The code

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fxds.simulator import (
    Simulator, Market, Trader, MarketParticipants, RiskLimits,
    TraderAction, SpotProcess, ParticipantBias,
    passive, over_trading, risk_reducing, run_many,
)
from fxds.plotting import (
    use_house_style, style_axis, mark_level,
    PRIMARY, SECONDARY, TERTIARY, QUATERNARY, MUTED, ALERT,
)

use_house_style()
pd.set_option("display.precision", 6)

### Task A — a ticking mid

Up or down by a fixed increment, fifty-fifty, once per tick. That's it.

In [2]:
sim = Simulator(market=Market(initial_spot=1.3000, spot_increment=0.0005), seed=1)
sim.run(150)
path = sim.to_frame()

fig, ax = plt.subplots()
ax.plot(path["step"], path["mid"], color=PRIMARY, linewidth=1.6)
mark_level(ax, 0, "")
ax.axhline(1.3000, color=MUTED, linestyle="--", linewidth=1)
style_axis(
    ax, "Task A — a random walk in spot", "Tick", "Mid (CCY2 per CCY1)",
    "Fifty-fifty up or down by a fixed increment. No drift, no trend — any pattern you see is your eyes, not the data.",
)
plt.show()

print(f"start {sim.market.initial_spot:.4f}   end {sim.market.mid:.4f}   "
      f"after {sim.market.step_count} ticks")

start 1.3000   end 1.2980   after 150 ticks


### Task B — a two-way price, and what crossing it costs

Now add the spread and trade on it. The cleanest demonstration is the one that removes luck entirely: **buy, watch spot move in your favour by exactly the spread, then sell back.**

In [3]:
from fxds.simulator import Trader as T

SPREAD = 0.0010       # a 10-pip spread
HALF = SPREAD / 2     # 5 pips each side of the mid

print("How far does spot have to move before a round trip breaks even?\n")
print(f"{'spot move (pips)':>18} {'P&L (pips)':>12}   verdict")
for move_pips in [0, 2, 5, 8, 10, 12, 15]:
    trader = T()
    trader.take_price(TraderAction.BUY, HALF)      # pay the offer
    trader.mark_to_market(move_pips * 0.0001)      # spot moves in your favour
    trader.take_price(TraderAction.SELL, HALF)     # give the bid to close
    pnl_pips = trader.pnl / 0.0001
    verdict = "break even" if abs(pnl_pips) < 1e-9 else ("profit" if pnl_pips > 0 else "loss")
    print(f"{move_pips:>18} {pnl_pips:>12.1f}   {verdict}")

print("\nThe break-even point is a FULL spread, not half of one:")
print("  5 pips paid lifting the offer + 5 pips paid giving the bid = 10 pips to recover.")
print("\nSpot moving your way by 8 pips still loses money. That is the structural")
print("negative carry of price taking, with no randomness involved at all.")

How far does spot have to move before a round trip breaks even?

  spot move (pips)   P&L (pips)   verdict
                 0        -10.0   loss
                 2         -8.0   loss
                 5         -5.0   loss
                 8         -2.0   loss
                10          0.0   break even
                12          2.0   profit
                15          5.0   profit

The break-even point is a FULL spread, not half of one:
  5 pips paid lifting the offer + 5 pips paid giving the bid = 10 pips to recover.

Spot moving your way by 8 pips still loses money. That is the structural
negative carry of price taking, with no randomness involved at all.


That is the structural negative carry, with no randomness involved. The market did exactly what you wanted and it still wasn't enough — you needed it to move *more than a full spread* just to break even.

### Task C — price making, the mirror image

In [4]:
maker = T()
print("The market buys from you (so you SOLD — position goes shorter):")
maker.make_price(market_buys=True, notional=1.0, half_spread=HALF)
print(f"   position {maker.position:+.0f}   P&L {maker.pnl:+.5f}   ← you EARNED half a spread")

print("\nThen the market sells to you (offsetting flow — Ch. 3, scenario 1):")
maker.make_price(market_buys=False, notional=1.0, half_spread=HALF)
print(f"   position {maker.position:+.0f}   P&L {maker.pnl:+.5f}   ← flat, and up the FULL spread")

print("\nSame two trades, opposite sign to the price taker. The difference is only")
print("who initiated — not which way the trade went.")

The market buys from you (so you SOLD — position goes shorter):
   position -1   P&L +0.00050   ← you EARNED half a spread

Then the market sells to you (offsetting flow — Ch. 3, scenario 1):
   position +0   P&L +0.00100   ← flat, and up the FULL spread

Same two trades, opposite sign to the price taker. The difference is only
who initiated — not which way the trade went.


> **The sign convention that trips people up.** "Market buys" means the market bought **from you**, so *your* position gets **shorter**. Reading it as "the market is a buyer so I'm long" gets the sign backwards on every price-making trade. `tests/test_simulator.py` asserts it explicitly for exactly this reason.

### A full session — Tasks A, B and C together

In [5]:
sim = Simulator(
    market=Market(initial_spot=1.3000, spot_increment=0.0005, bid_offer_spread=0.0010),
    participants=MarketParticipants(buy_probability=0.15, sell_probability=0.15),
    seed=7,
)
frame = sim.run(300, risk_reducing(max_position=6.0))

fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1.3, 1.3]})

axes[0].fill_between(frame["step"], frame["bid"], frame["offer"],
                     color=MUTED, alpha=0.18, label="bid–offer")
axes[0].plot(frame["step"], frame["mid"], color=PRIMARY, linewidth=1.6, label="mid")
axes[0].legend(loc="upper left")
style_axis(axes[0], "A full session — price making with a position limit", "",
           "Spot (CCY2 per CCY1)", "")

axes[1].step(frame["step"], frame["position"], color=SECONDARY, linewidth=1.6, where="post")
axes[1].axhline(0, color=MUTED, linewidth=0.8)
for lim in (6, -6):
    axes[1].axhline(lim, color=ALERT, linestyle=":", linewidth=1.1)
style_axis(axes[1], "", "", "Position (CCY1)", "")

pnl_colour = TERTIARY if frame["pnl"].iloc[-1] >= 0 else ALERT
axes[2].fill_between(frame["step"], frame["pnl"], color=pnl_colour, alpha=0.35)
axes[2].plot(frame["step"], frame["pnl"], color=pnl_colour, linewidth=1.6)
axes[2].axhline(0, color=MUTED, linewidth=0.8)
style_axis(axes[2], "", "Tick", "P&L (CCY2)",
           "Dotted red lines are the position limit. The trader only crosses the spread to trim back inside it.")

plt.tight_layout(); plt.show()

s = sim.summary
print(f"ticks {s['ticks']}   final position {s['position']:+.0f}   P&L {s['pnl']:+.5f}")
print(f"trades taken {s['trades_taken']} (spread paid {s['spread_paid']:.5f})")
print(f"trades made  {s['trades_made']} (spread earned {s['spread_earned']:.5f})")
print(f"net spread   {s['spread_earned'] - s['spread_paid']:+.5f}")

ticks 300   final position -5   P&L +0.01750
trades taken 7 (spread paid 0.00350)
trades made  100 (spread earned 0.05000)
net spread   +0.04650


### The Streamlit app

Everything above runs headlessly. The interactive version is the same engine with a front end:

```bash
streamlit run fxds/simulator/app.py
```

Sliders for the tick rate, spread and increment; Go/Pause and Stop; Buy and Sell buttons; toggles for every extension. The book suggests starting at **five seconds between ticks** while the interaction between market, position and P&L becomes familiar. That's good advice — the mechanics are obvious at five seconds and a blur at one.

## The experiment the book doesn't run

Chapter 3 says *don't over-trade when there is spread cross involved*. A single session can't show that — you'll have sessions where the over-trader gets lucky. The way to see a structural effect is to run it a thousand times.

Two strategies, same markets, same client flow:

- **Passive** — never crosses the spread. Takes whatever position the flow hands it.
- **Over-trading** — crosses the spread at random on half of all ticks. No view, no edge, just activity.

Each pair of sessions shares a seed, so both face an **identical** spot path and identical flow. The strategy's own randomness comes from a separate generator precisely so it can't disturb the market — without that the comparison would be worthless.

In [6]:
SESSIONS, TICKS = 1000, 250
flow = dict(buy_probability=0.15, sell_probability=0.15)

quiet = run_many(SESSIONS, TICKS, passive, base_seed=0, participant_kwargs=flow)
busy = run_many(SESSIONS, TICKS, over_trading(0.5), base_seed=0, participant_kwargs=flow)

summary = pd.DataFrame({
    "passive": [quiet["pnl"].mean(), quiet["pnl"].median(), quiet["pnl"].std(),
                (quiet["pnl"] > 0).mean(), quiet["spread_paid"].mean(),
                quiet["spread_earned"].mean(), quiet["trades_taken"].mean()],
    "over-trading": [busy["pnl"].mean(), busy["pnl"].median(), busy["pnl"].std(),
                     (busy["pnl"] > 0).mean(), busy["spread_paid"].mean(),
                     busy["spread_earned"].mean(), busy["trades_taken"].mean()],
}, index=["mean P&L", "median P&L", "P&L std dev", "win rate",
          "spread paid", "spread earned", "trades taken"])

print(f"{SESSIONS} sessions of {TICKS} ticks each, identical markets\n")
print(summary.to_string(float_format=lambda v: f"{v:+.5f}"))

1000 sessions of 250 ticks each, identical markets

               passive  over-trading
mean P&L      +0.03699      -0.02521
median P&L    +0.03675      -0.02550
P&L std dev   +0.04702      +0.08050
win rate      +0.83900      +0.31900
spread paid   +0.00000      +0.06263
spread earned +0.03764      +0.03764
trades taken  +0.00000    +125.26900


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

bins = np.linspace(min(quiet["pnl"].min(), busy["pnl"].min()),
                   max(quiet["pnl"].max(), busy["pnl"].max()), 55)
axes[0].hist(quiet["pnl"], bins=bins, color=PRIMARY, alpha=0.62, label="passive")
axes[0].hist(busy["pnl"], bins=bins, color=SECONDARY, alpha=0.62, label="over-trading")
axes[0].axvline(quiet["pnl"].mean(), color=PRIMARY, linestyle="--", linewidth=1.8)
axes[0].axvline(busy["pnl"].mean(), color=SECONDARY, linestyle="--", linewidth=1.8)
axes[0].axvline(0, color=MUTED, linewidth=1)
axes[0].legend()
style_axis(axes[0], "P&L distribution over 1,000 sessions", "Session P&L (CCY2)",
           "Number of sessions",
           "Dashed lines are the means. Both spread out, but the whole over-trading distribution sits to the left.")

paired = busy["pnl"].to_numpy() - quiet["pnl"].to_numpy()
axes[1].hist(paired, bins=50, color=QUATERNARY, alpha=0.75)
axes[1].axvline(0, color=MUTED, linewidth=1)
axes[1].axvline(paired.mean(), color=ALERT, linestyle="--", linewidth=1.8)
style_axis(axes[1], "Paired difference, session by session", "Over-trading P&L − passive P&L (CCY2)",
           "Number of sessions",
           "Same market, same flow, only the trading differs. Almost entirely left of zero — this is the cost of activity.")

plt.tight_layout(); plt.show()

print(f"mean paired difference   {paired.mean():+.5f}")
print(f"sessions where over-trading did worse   {(paired < 0).mean():.1%}")
print(f"mean spread paid by the over-trader     {busy['spread_paid'].mean():.5f}")

mean paired difference   -0.06221
sessions where over-trading did worse   89.1%
mean spread paid by the over-trader     0.06263


The paired chart is the one that settles it. Because each pair faces an identical market, the difference between them **isolates the trading decision** — and it is almost entirely negative, centred close to the spread paid.

Note what it is *not*: it isn't that over-trading always loses. The distribution has a right tail; plenty of individual sessions come out ahead. The point is that the whole distribution has been shifted left by a predictable amount, and no amount of activity shifts it back. That's what "structural" means.

## Experiments

### Experiment 1 — Does trading twice as often cost twice as much?

**Predict:** the expected cost is `n × s/2`. So doubling the trade frequency should double the spread bill. Does the *P&L gap* double too, or does something else interfere?

In [8]:
rows = []
for p in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
    out = run_many(400, 200, over_trading(p) if p > 0 else passive,
                   base_seed=0, participant_kwargs=flow)
    rows.append({"P(trade)": p, "trades": out["trades_taken"].mean(),
                 "spread paid": out["spread_paid"].mean(),
                 "mean P&L": out["pnl"].mean(), "P&L std": out["pnl"].std()})
freq = pd.DataFrame(rows)
print(freq.to_string(index=False, float_format=lambda v: f"{v:+.5f}"))

fig, ax = plt.subplots()
ax.plot(freq["P(trade)"], freq["mean P&L"], "o-", color=PRIMARY, label="mean P&L")
ax.plot(freq["P(trade)"], -freq["spread paid"], "s--", color=ALERT, label="−(spread paid)")
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.legend()
style_axis(ax, "The cost of activity is linear in how often you trade",
           "Probability of trading on a given tick", "CCY2",
           "The two lines track each other: the P&L drag IS the spread crossed. Nothing else is going on.")
plt.show()

 P(trade)     trades  spread paid  mean P&L  P&L std
 +0.00000   +0.00000     +0.00000  +0.02731 +0.03546
 +0.10000  +20.05250     +0.01003  +0.01713 +0.04282
 +0.25000  +49.70750     +0.02485  +0.00438 +0.05031
 +0.50000  +99.80000     +0.04990  -0.02204 +0.06161
 +0.75000 +149.65000     +0.07483  -0.04973 +0.07567
 +1.00000 +200.00000     +0.10000  -0.07539 +0.07582


**Result:** the spread bill is dead linear in trade frequency, and the mean P&L falls in lockstep. The two lines are nearly parallel — the entire drag is spread cross, exactly as the expectation formula says.

Notice the P&L standard deviation barely moves. Over-trading doesn't buy you more upside in exchange for the cost. **You pay and get nothing.**

### Experiment 2 — Skewed flow: what happens when the market only sells to you?

Chapter 3's scenario 3 is the price maker's nightmare: flow that all comes one way while the market moves against you.

**Predict:** with participants who only ever sell to you, you get progressively longer. If spot is a fair coin, does the spread you earn cover it?

In [9]:
rows = []
for label, kw in [
    ("balanced  (15% / 15%)", dict(buy_probability=0.15, sell_probability=0.15)),
    ("mild skew (20% / 10%)", dict(buy_probability=0.20, sell_probability=0.10)),
    ("one-way   (30% /  0%)", dict(buy_probability=0.30, sell_probability=0.00)),
]:
    out = run_many(600, 250, passive, base_seed=0, participant_kwargs=kw)
    rows.append({"flow": label, "mean P&L": out["pnl"].mean(),
                 "P&L std": out["pnl"].std(),
                 "mean |final position|": out["position"].abs().mean(),
                 "spread earned": out["spread_earned"].mean()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

                 flow  mean P&L  P&L std  mean |final position|  spread earned
balanced  (15% / 15%)   +0.0367  +0.0454                +6.6517        +0.0375
mild skew (20% / 10%)   +0.0363  +0.1190               +25.1350        +0.0375
one-way   (30% /  0%)   +0.0299  +0.3281               +75.0583        +0.0375


**Result:** the mean P&L stays near the spread earned in every case — spot is a fair coin, so the accumulated position is a fair bet in expectation. But look at the **standard deviation** and the **average final position**. One-way flow leaves you carrying a large position, and the P&L swings scale with it.

That's the price maker's real problem, and it isn't about expected value. You earn the spread reliably and you carry risk you didn't choose. Chapter 3's advice — sit and wait for offsetting flow, and trim only when the position gets uncomfortable — is a variance-management strategy, not a profit-maximising one.

### Experiment 3 — Risk limits that don't line up

Chapter 3: risk limits and P&L targets **should be in line**. Greater risk offers greater reward but *guarantees* only greater P&L volatility.

**Predict:** you cap the position at 3 units instead of 20 but keep the same ambitious P&L target. What happens to the fraction of sessions that reach the target — and, less obviously, what happens to the fraction that hit the *stop loss*?

In [10]:
# The trader's own cap has to match the limit for the limit to mean anything -
# price-making flow deliberately ignores it (you cannot decline a trade that has
# already happened), so the trader must trim to stay inside it.
rows = []
for label, cap, target in [
    ("aligned:   cap 20, target 0.05", 20.0, 0.05),
    ("aligned:   cap  3, target 0.01",  3.0, 0.01),
    ("MISALIGNED: cap  3, target 0.05",  3.0, 0.05),
]:
    out = run_many(
        600, 400, risk_reducing(max_position=cap), base_seed=0,
        trader_kwargs=dict(limits=RiskLimits(max_position=cap * 2,
                                             stop_loss=-target, profit_target=target)),
        participant_kwargs=flow,
    )
    stops = out["stopped"].value_counts(normalize=True)
    rows.append({"configuration": label,
                 "hit target": stops.get("profit target", 0.0),
                 "hit stop": stops.get("stop loss", 0.0),
                 "ran out of ticks": stops.get("none", 0.0),
                 "mean |position|": out["position"].abs().mean(),
                 "mean P&L": out["pnl"].mean()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

                  configuration  hit target  hit stop  ran out of ticks  mean |position|  mean P&L
 aligned:   cap 20, target 0.05      0.7783    0.0900            0.1317           7.1933    0.0380
 aligned:   cap  3, target 0.01      0.9433    0.0550            0.0017           2.2283    0.0092
MISALIGNED: cap  3, target 0.05      0.6167    0.0000            0.3833           2.0650    0.0431


**Result:** not what I predicted, and the data is more interesting than the guess was.

The tight cap still reaches the big target **62%** of the time. A price maker earns spread on every trade the flow brings, and over 400 ticks that income accumulates regardless of how small the position is — so the target is not unreachable at all. What actually changes is the *rate*:

| | cap 20 | cap 3 |
|---|---|---|
| hit the 0.05 target | 78% | 62% |
| **hit the stop loss** | **9%** | **0%** |
| ran out of ticks | 13% | 38% |

Two effects, pulling opposite ways:

- The tight cap **caps the downside too**. It never once hit the stop loss, where the wide cap did 9% of the time. That is the limit doing exactly its job.
- But it also **slows everything down**. Three times as many sessions simply ran out of ticks without resolving either way.

So the honest version of Chapter 3's point is not "a tight limit makes the target impossible". It's that **a position limit sets the speed at which P&L can move in either direction, and a target has to be sized to what that speed can deliver within your horizon.** Pair a tight limit with a target calibrated for a wide one and you haven't created an impossible objective — you've created one that mostly resolves as "time expired", which is a worse kind of uninformative.

Shorten the horizon in the cell above from 400 ticks to 100 and the picture changes completely: the wide cap hits the target 5% of the time and the tight cap **0%**. Misalignment is a relationship between limit, target *and* horizon — not between limit and target alone.

In [11]:
rows = []
configs = {
    "fixed increment, independent flow": (
        dict(), dict(buy_probability=0.15, sell_probability=0.15)),
    "volatility spot, independent flow": (
        dict(process=SpotProcess.VOLATILITY, volatility=0.12),
        dict(buy_probability=0.15, sell_probability=0.15)),
    "volatility spot, mean-reverting flow": (
        dict(process=SpotProcess.VOLATILITY, volatility=0.12),
        dict(buy_probability=0.15, sell_probability=0.15,
             bias=ParticipantBias.MEAN_REVERTING, bias_strength=0.6)),
}
for label, (mkt, flw) in configs.items():
    q = run_many(500, 250, passive, base_seed=0, market_kwargs=mkt, participant_kwargs=flw)
    b = run_many(500, 250, over_trading(0.5), base_seed=0, market_kwargs=mkt, participant_kwargs=flw)
    rows.append({"configuration": label,
                 "passive mean": q["pnl"].mean(),
                 "over-trading mean": b["pnl"].mean(),
                 "gap": b["pnl"].mean() - q["pnl"].mean()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:+.5f}"))

                       configuration  passive mean  over-trading mean      gap
   fixed increment, independent flow      +0.03516           -0.02412 -0.05927
   volatility spot, independent flow      +0.03648           -0.03094 -0.06742
volatility spot, mean-reverting flow      +0.02386           -0.04356 -0.06742


**Result:** the gap survives every configuration. It has to — it's a cost imposed at the moment of trading, and nothing about how spot evolves or how flow arrives can refund it.

Look closely and the last two rows have an **identical** gap, to five decimal places, despite quite different mean P&Ls. That isn't a coincidence or a bug. Adding a flow bias changes the position path, but it changes it *the same way for both strategies* — they face identical participants. So the bias cancels in the difference, leaving only what actually distinguishes them: the spread the over-trader crossed.

That cancellation is the paired design earning its keep. It's also a good reason to prefer paired comparisons whenever you can construct one — the noise you remove is noise you don't have to average away with more sessions.

This generalises well beyond the simulator. Transaction costs are not noise you can trade your way out of. They're a fixed toll, and the only control you have is how often you pay it.

## Common misconceptions

**"The bid–offer spread is what the market maker charges."**
It's what the market maker is *paid for taking risk*. Chapter 3 is explicit: the spread exists to cover holding the position until an offsetting trade arrives. It's compensation for uncertainty, not a fee.

**"'Market buys' means I'm long."**
It means the market bought **from you**, so you're **shorter**. Backwards on every price-making trade if you get this wrong.

**"Being a market maker is free money."**
You collect the spread but you don't choose the position. Chapter 3's scenario 3 has the trader stuck short as the market runs higher — spread collected, and a loss much bigger than the spread. Experiment 2 shows the variance cost.

**"If I trade more I have more chances to be right."**
Only if you have a view. With no edge, more trades means only more spread paid — Experiment 1 measures it, and the P&L standard deviation barely moves, so you don't even buy upside with the cost.

**"A tight risk limit is always the cautious choice."**
Only if the P&L target moves with it. Experiment 3's misaligned case has a target its own position limit makes arithmetically unreachable.

**"The order of operations in a tick is an implementation detail."**
It changes the P&L. Marking the position *before* processing the trade is what stops a trade from earning a move that happened before it existed. Get it backwards and your simulator flatters every trade by one tick of drift.

**"A single session tells you whether a strategy works."**
It doesn't. The passive session in the walkthrough above lost money — because spot trended while it was short, not because passivity is wrong. That's why the experiment runs a thousand paired sessions.

## Check yourself

1. Spot is 1.3000 with a 10-pip spread. You buy, spot rises 4 pips, you sell. What's your P&L in pips?
2. The market buys 5 units from you at your offer. Which way did your position move, and what happened to your P&L?
3. You run 500 ticks trading at random on 20% of them, with a 10-pip spread and 1-unit trades. Roughly what's your expected P&L?
4. Your position limit is 2 units and your profit target needs a 10-unit position to reach. What's wrong?
5. Why does the passive strategy still have a wide P&L distribution when it never crosses a spread?

In [12]:
#@title Answers — run this cell to reveal
from IPython.display import Markdown
Markdown(r'''
**1.** **−6 pips.** You paid 5 pips of spread going in and 5 coming out (10 total) and made 4 pips on the move. Spot moved your way and you still lost, because it didn't move by more than the round-trip cost.

**2.** Your position went **shorter by 5** — they bought from you, so you sold. Your P&L went **up** by 5 × half-spread. You earned the spread and inherited a short you didn't choose.

**3.** About 100 trades × 5 pips = **−500 pips**, or −0.05 in rate terms. The market's direction contributes nothing in expectation; the spread contributes all of it. Experiment 1's linear relationship is this calculation.

**4.** They're **misaligned** — the target is unreachable under the limit, so the only possible outcomes are "ran out of time" or "stopped out". Chapter 3: limits and targets must be in line. Greater risk offers the *opportunity* for greater reward; a small limit removes the opportunity while the target pretends it exists.

**5.** Because it still **carries a position**. Price making hands you inventory whether you want it or not, and that position is marked to every spot move. You avoid the spread cost, not the market risk. Experiment 2 shows the variance rising sharply as flow gets more one-sided.
''')


**1.** **−6 pips.** You paid 5 pips of spread going in and 5 coming out (10 total) and made 4 pips on the move. Spot moved your way and you still lost, because it didn't move by more than the round-trip cost.

**2.** Your position went **shorter by 5** — they bought from you, so you sold. Your P&L went **up** by 5 × half-spread. You earned the spread and inherited a short you didn't choose.

**3.** About 100 trades × 5 pips = **−500 pips**, or −0.05 in rate terms. The market's direction contributes nothing in expectation; the spread contributes all of it. Experiment 1's linear relationship is this calculation.

**4.** They're **misaligned** — the target is unreachable under the limit, so the only possible outcomes are "ran out of time" or "stopped out". Chapter 3: limits and targets must be in line. Greater risk offers the *opportunity* for greater reward; a small limit removes the opportunity while the target pretends it exists.

**5.** Because it still **carries a position**. Price making hands you inventory whether you want it or not, and that position is marked to every spot move. You avoid the spread cost, not the market risk. Experiment 2 shows the variance rising sharply as flow gets more one-sided.


## Where next

**Notebook 03 — Chapter 4** covers market structure: how the interbank broker market actually works, what a "specific" is, and why a direct call is a different proposition from working an interest through a broker. Conceptual, no practical, and short.

Then Chapter 5 and Practical B, where the underlying stops being a random walk you watch and becomes a distribution you price against.

Before moving on, run the app for a while:

```bash
streamlit run fxds/simulator/app.py
```

Turn price making on, set both probabilities to 15%, and try to end a 200-tick session flat and up. It is harder than the arithmetic suggests, and that difficulty is the lesson.